# Week 3: Data Preprocessing

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import re
import datetime

In [2]:
df = pd.read_csv("../../data/New/df_messy.csv")
df.info()

C:\Users\ahyo\AppData\Local\Temp\ipykernel_23244\2750436862.py:1: DtypeWarning: Columns (2,45,78,79,80,81) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../../data/New/df_messy.csv")


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 399157 entries, 0 to 399156
Data columns (total 82 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   Flooring                      256352 non-null  object 
 1   ViewYN                        359640 non-null  object 
 2   WaterfrontYN                  175 non-null     object 
 3   BasementYN                    9747 non-null    object 
 4   PoolPrivateYN                 358610 non-null  object 
 5   OriginalListPrice             398378 non-null  float64
 6   ListingKey                    399157 non-null  int64  
 7   ListAgentEmail                398296 non-null  object 
 8   CloseDate                     399157 non-null  object 
 9   ClosePrice                    399155 non-null  float64
 10  ListAgentFirstName            395752 non-null  object 
 11  ListAgentLastName             399120 non-null  object 
 12  Latitude                      398964 non-nul

In [3]:
df['Stories'].unique()

array([nan,  1.,  2.])

There are so few listings in states other than California that it makes sense remove those rows entirely

In [4]:
def percentage_drop(before, after):
    print(f"Before: {before}")
    print(f"After: {after} (-{before-after} rows)")
    print(f"% change: -{((before-after)/before)*100:.2f}")

In [25]:
print("----------------CLEANING----------------")

# first copy df
df_cleaned = df.copy()
before = len(df_cleaned)
print(f"Original Number of Rows: {before}")

###################################################
# 1: Identify Impossible Values and Drop Duplicates
###################################################

# drop duplicates
print("\n----- Dropping Duplicates -----")
print(f"{df.duplicated().sum()} duplicate rows found")
df_cleaned = df_cleaned.drop_duplicates()
after = len(df_cleaned)
percentage_drop(before, after)
before = after

# drop rows with impossible or na values for the target variable ClosePrice
print("\n----- Dropping Rows with NaN or Impossible ClosePrice Values -----")
print(f"{(df_cleaned['ClosePrice'] <= 1000).sum() + df_cleaned['ClosePrice'].isna().sum()} impossible or NaN values round")
df_cleaned = df_cleaned[df_cleaned['ClosePrice'] > 1000]
df_cleaned = df_cleaned.dropna(subset=['ClosePrice'])
after = len(df_cleaned)
percentage_drop(before, after)
before = after

# convert impossible coordinates to NaN
invalid_coords = (
    (df_cleaned['Latitude'] < -90) | (df_cleaned['Latitude'] > 90) |
    (df_cleaned['Longitude'] < -180) | (df_cleaned['Longitude'] > 180) |
    (df_cleaned['Latitude'] == 0) | (df_cleaned['Longitude'] == 0)
)
df_cleaned.loc[invalid_coords,['Latitude','Longitude']] = np.nan

# LivingArea
df_cleaned.loc[df_cleaned['LivingArea'] <= 0, 'LivingArea'] = np.nan
# df_cleaned.loc[df_cleaned['LivingArea'] > 20000, 'LivingArea'] = np.nan

# ParkingTotal
df_cleaned.loc[df_cleaned['ParkingTotal'] < 0, 'ParkingTotal'] = np.nan
df_cleaned.loc[df_cleaned['ParkingTotal'] > 20, 'ParkingTotal'] = np.nan

# LotSizeSquareFeet
df_cleaned.loc[df_cleaned['LotSizeSquareFeet'] <= 0, 'LotSizeSquareFeet'] = np.nan
# df_cleaned.loc[df_cleaned['LotSizeSquareFeet'] > 500000, 'LotSizeSquareFeet'] = np.nan

# YearBuilt
df_cleaned.loc[df_cleaned['YearBuilt'] < 1850, 'YearBuilt'] = np.nan

# GarageSpaces
df_cleaned.loc[df_cleaned['GarageSpaces'] >= 10, 'GarageSpaces'] = np.nan

#################################################################
# 2: Drop Columns with > 60% NaN Values and single-valued columns
#################################################################

# keep columns with at least 60% non-null values
print("\n----- Dropping Columns with > 40% null count -----")
print(f"Dropping {(df_cleaned.isna().mean() > 0.4).sum()} columns...")
df_cleaned = df_cleaned.dropna(thresh=len(df_cleaned) * 0.60, axis=1)
print(f"New column count: {len(df_cleaned.columns)}")

# drop states other than CA
df_cleaned = df_cleaned[df_cleaned['StateOrProvince'] == 'CA']

# Drop columns with a single value
print("\n----- Dropping Columns with Only 1 value -----")
print(f"Dropping {len(df_cleaned.columns[df_cleaned.nunique()==1])} columns...")
single_val_cols = df_cleaned.columns[df_cleaned.nunique() == 1]
df_cleaned = df_cleaned.drop(columns=single_val_cols)
print(f"New columns count: {len(df_cleaned.columns)}")

###############################################
# 3: Change DateTime columns to DateTime format
###############################################

date_cols = df_cleaned.filter(regex='(?i)Date').columns.tolist()
for col in date_cols:
    df_cleaned[col] = pd.to_datetime(df_cleaned[col], errors='coerce')

################################
# 4: Drop Other Columns Manually
################################

print("\n----- Dropping Redundant, Unhelpful, and Leaky Columns -----")
# Redundant columns
redundant_cols = [
    'ListingKeyNumeric',
    'ListingId',
    'ListAgentFullName',
    'LotSizeAcres',
    'LotSizeArea'
]

# Columns with no predictive power
unhelpful_cols = [
    'ListAgentEmail',
    'ListAgentFirstName',
    'ListAgentLastName',
    'ListOfficeName',
    'BuyerOfficeName',
    'BuyerAgentMlsId',
    'BuyerAgentFirstName',
    'BuyerAgentLastName',
    'UnparsedAddress',
    'StreetNumberNumeric'
]

# Columns that leak information
leaky_cols = [
    'ListPrice',
    'OriginalListPrice',
    'PurchaseContractDate',
    'DaysOnMarket'
]

print(f"Dropping {len(redundant_cols)} redundant columns, {len(unhelpful_cols)} unhelpful columns, and {len(leaky_cols)} leaky columns")
df_cleaned = df_cleaned.drop(columns=redundant_cols + leaky_cols + unhelpful_cols)
print(f"New columns count: {len(df_cleaned.columns)}")

##########################################################
# 5: Convert YN Columns to Boolean and Impute NaNs with -1
##########################################################

bool_cols = ['ViewYN', 'PoolPrivateYN', 'AttachedGarageYN', 'FireplaceYN', 'NewConstructionYN']

for col in bool_cols:
    df_cleaned[col] = df_cleaned[col].fillna('-1')
    
df_cleaned[bool_cols] = df_cleaned[bool_cols].astype(int)

##########################
# 6: Impute Missing Values
##########################

unknown_fill = ['Flooring', 'MLSAreaMajor', 'BuyerOfficeAOR', 'City', 'Levels', 'HighSchoolDistrict',
                'PostalCode', 'BuyerAgentAOR', 'ListAgentAOR']

zero_fill = ['AssociationFee', 'GarageSpaces', 'ParkingTotal']

median_fill = ['LivingArea', 'LotSizeSquareFeet', 'YearBuilt']

mode_fill = ['BedroomsTotal', 'BathroomsTotalInteger', 'Stories']

for col in unknown_fill:
    df_cleaned[col] = df_cleaned[col].fillna('Unknown')

for col in zero_fill:
    df_cleaned[col] = df_cleaned[col].fillna(0)

for col in median_fill:
    df_cleaned[col] = df_cleaned[col].fillna(df_cleaned[col].median())

for col in mode_fill:
    df_cleaned[col] = df_cleaned[col].fillna(df_cleaned[col].mode()[0])

# Drop missing latitude and Longitude rows
print("\n----- Dropping Rows with NaN Latitude or Longitude -----")
print(f"Dropping {df_cleaned[['Latitude', 'Longitude']].isna().any(axis=1).sum()} rows...")
df_cleaned = df_cleaned.dropna(subset=['Latitude', 'Longitude'])
after = len(df_cleaned)
percentage_drop(before, after)
before = after

----------------CLEANING----------------
Original Number of Rows: 399157

----- Dropping Duplicates -----
31 duplicate rows found
Before: 399157
After: 399126 (-31 rows)
% change: -0.01

----- Dropping Rows with NaN or Impossible ClosePrice Values -----
9 impossible or NaN values round
Before: 399126
After: 399117 (-9 rows)
% change: -0.00

----- Dropping Columns with > 40% null count -----
Dropping 28 columns...
New column count: 54

----- Dropping Columns with Only 1 value -----
Dropping 4 columns...
New columns count: 50

----- Dropping Redundant, Unhelpful, and Leaky Columns -----
Dropping 5 redundant columns, 10 unhelpful columns, and 4 leaky columns
New columns count: 31

----- Dropping Rows with NaN Latitude or Longitude -----
Dropping 232 rows...
Before: 399117
After: 398858 (-259 rows)
% change: -0.06


In [16]:
df_cleaned.Levels.unique()

array(['ThreeOrMore', 'One', 'Unknown', 'Two', 'MultiSplit',
       'Two,ThreeOrMore', 'One,MultiSplit', 'One,Two', 'Two,MultiSplit',
       'One,ThreeOrMore', 'One,Two,MultiSplit', 'ThreeOrMore,MultiSplit',
       'One,Two,ThreeOrMore', 'Two,ThreeOrMore,MultiSplit',
       'One,Two,ThreeOrMore,MultiSplit', 'Two,One', 'MultiSplit,One',
       'ThreeOrMore,One', 'Two,MultiSplit,One'], dtype=object)

For now, I will keep nans in boolean valued columns because there is no easy interpretation of what a `nan` value means here

## Date distribution plots

#### CloseDate Distribution

In [ ]:
plt.figure()
sns.histplot(data=df_cleaned, x='CloseDate', bins = 100, kde=True)
plt.xticks(rotation=45)
plt.title('CloseDate Distribution')
plt.show()

#### Does `ClosePrice` have any relation with `CloseDate`?

In [ ]:
Q1 = df_cleaned['ClosePrice'].quantile(0.25)
Q3 = df_cleaned['ClosePrice'].quantile(0.75)
IQR = Q3 - Q1

df_cleaned_no_outliers = df_cleaned[~((df_cleaned['ClosePrice'] < (Q1 - 1.5 * IQR)) | (df_cleaned['ClosePrice'] > (Q3 + 1.5 * IQR)))]

In [ ]:
plt.figure()
sns.scatterplot(data=df_cleaned_no_outliers, x='CloseDate', y='ClosePrice', s=5)
plt.xticks(rotation=45)
plt.show()

## Creating training and testing sets

Export clean dataset so, I don't have to reclean everytime

In [26]:
df_cleaned.to_csv('../../data/New/df_cleaned.csv', index=False)

Function to split into training and testing by `CloseDate`

In [ ]:
from dateutil.relativedelta import relativedelta

def train_test_split_by_month(df, date_col='CloseDate', n_train_months=6):
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])

    # most recent month present in the data
    max_date = df[date_col].max()
    test_month_start = max_date.replace(day=1)
    test_month_end = test_month_start + relativedelta(months=1)

    # training window: the n_train_months immediately before the test month
    train_start = test_month_start - relativedelta(months=n_train_months)
    train_end = test_month_start

    test_df = df[(df[date_col] >= test_month_start) & (df[date_col] < test_month_end)]
    train_df = df[(df[date_col] >= train_start) & (df[date_col] < train_end)]

    print(f"Test month: {test_month_start.date()} to {test_month_end.date()} ({len(test_df)} rows)")
    print(f"Train window: {train_start.date()} to {train_end.date()} ({len(train_df)} rows)")

    return train_df, test_df

# example usage
train_df, test_df = train_test_split_by_month(df_cleaned, date_col='CloseDate', n_train_months=6)

## Cleaned Data Sanity Checks

In [ ]:
n = len(df_cleaned)

# 1) Missingness ranked — verifies my counts above
print("-----Ranked Missingness-----")
miss = (df_cleaned.isna().sum().to_frame("n_missing")
          .assign(pct=lambda d: (d.n_missing / n * 100).round(2), dtype=df_cleaned.dtypes.astype(str))
          .sort_values("n_missing", ascending=False))
print(miss[miss.n_missing > 0])

# 2) Cardinality of object cols → spot multi-value / free-text / PII
print("\n-----Number of Unique Entries-----")
print(df_cleaned.select_dtypes("object").nunique().sort_values(ascending=False))
print("\n-----Cardinality of Object Columns-----")
for c in ["Flooring","Levels","MLSAreaMajor","HighSchoolDistrict","ListAgentAOR"]:
    print(f"\n--- {c} ---\n{df_cleaned[c].value_counts(dropna=False).head(5)}")

# 3) Zeros / negatives / impossible values
print("\n-----Impossible Values-----")
num = df_cleaned.select_dtypes("number").columns
print(df_cleaned[num].describe().T[["min","25%","50%","75%","max"]])
print("\nzeros:\n", (df_cleaned[num] == 0).sum().sort_values(ascending=False))
print("\nnegatives:\n", (df_cleaned[num] < 0).sum().sort_values(ascending=False))

# 5) Sanity checks on the modeling fields
print("\nClosePrice <= 1000 :", (df_cleaned.ClosePrice <= 1000).sum())
print("LivingArea <= 0    :", (df_cleaned.LivingArea <= 0).sum())
print("YearBuilt < 1800   :", (df_cleaned.YearBuilt < 1800).sum())
print("Lat/Lon == 0       :", ((df_cleaned.Latitude == 0) | (df_cleaned.Longitude == 0)).sum())

# 6) PostalCode format
print("\n-----Postal Code Formats-----")
print(df_cleaned.PostalCode.astype(str).str.len().value_counts())

# 7) Duplicates
print("\n-----Duplicates-----")
print("dupe rows:", df_cleaned.duplicated().sum(), "| dupe keys:", df_cleaned.ListingKey.duplicated().sum())

### Notes:
1. `LotSizeArea` has no units (`LotSizeUnits` does not exist), so I will prefer `LotSizeSquareFeet`
2. Boolean features are left with NaN values in as these mean something other than True or False
3. For now, impossible values are only cutoff under the minimum not over the maximum
4. `PostalCode`'s have not been standardized
5. No numerical features have been normalized, but will be for building the `LinearRegression` model next week